# Adaptive Intent Alignment (Self-Aware RAG)

> Achieving **100% Recall Certainty** through logical smelting and domain anchoring.

### 🛠️ Architecture Overview

1. **Knowledge Smelting**: Background processing to extract the logical boundaries (Global Intent Mirror) of the knowledge base.
2. **Adaptive Domain Anchoring**: Dynamically resolving semantic ambiguity (e.g., "Agent" as AI vs. Bio) before retrieval.

---


In [2]:
import os
import dotenv
import bs4
import chromadb
import threading
import time
import json
from loguru import logger
from flashboot_core.utils import project_utils

root_path = project_utils.get_root_path()
dotenv.load_dotenv(os.path.join(root_path, ".env"), override=True)

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]

In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field

USER_AGENT environment variable not set, consider setting it to identify your requests.


## Phase 0: Dataset Collision Prep

We will load two conflicting domains into a single collection to test the disambiguation capability.

```mermaid
graph TD
    subgraph "Collision Zone"
        A[Domain: Biological] <--> B{Semantic Ambiguity: 'Agent'} <--> C[Domain: Technical]
    end
    style B fill:#f96,stroke:#333,stroke-width:4px
    style A fill:#dfd,stroke:#090
    style C fill:#ddf,stroke:#009
```


In [ ]:
# Domain A: Biological (Wikipedia/Biosecurity)
biological_urls = [
    "https://en.wikipedia.org/wiki/Biological_agent",
    "https://en.wikipedia.org/wiki/Biosecurity",
    "https://en.wikipedia.org/wiki/Infection_prevention_and_control",
]

# Domain B: Technical (Lilian Weng's Blog)
technical_urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
]

def load_and_split(urls, domain_label):
    loader = WebBaseLoader(urls)
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = splitter.split_documents(docs)
    for s in splits:
        s.metadata["domain"] = domain_label
    return splits

logger.info("Loading Domain A (Biological)...")
bio_splits = load_and_split(biological_urls, "biological")

logger.info("Loading Domain B (Technical)...")
tech_splits = load_and_split(technical_urls, "technical")

all_splits = bio_splits + tech_splits
logger.info(f"Total splits for collision test: {len(all_splits)}")


2026-01-27 16:24:26.780 | INFO     | __main__:<module>:24 - Loading Domain A (Technical)...
2026-01-27 16:24:34.042 | INFO     | __main__:<module>:27 - Loading Domain B (Biological)...
2026-01-27 16:24:40.324 | INFO     | __main__:<module>:31 - Total splits for collision test: 331


In [ ]:
embedding = OpenAIEmbeddings(
    model="text-embedding-v3",
    openai_api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
    check_embedding_ctx_length=False,
    chunk_size=10,
)

collection_name = "collision_v1"
local_host = "127.0.0.1"
local_port = 18000
chroma_client = chromadb.HttpClient(host=local_host, port=local_port)

# Forced Rebuild for the first test
if collection_name in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(collection_name)

vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=embedding,
    client=chroma_client,
    collection_name=collection_name,
)
logger.success(
    f"Collision collection '{collection_name}' initialized with dual domains."
)

2026-01-27 16:25:54.628 | SUCCESS  | __main__:<module>:24 - Collision collection 'collision_v1' initialized with dual domains.


In [12]:
# Collision Diagnostic: Combining Status Judgment & Evidence Chain
def run_collision_diagnostic(queries, vectorstore, k=5):
    for q in queries:
        results = vectorstore.similarity_search(q, k=k)
        tech_hits = [d for d in results if d.metadata.get("domain") == "technical"]
        bio_hits = [d for d in results if d.metadata.get("domain") == "biological"]

        tech_p = (len(tech_hits) / k) * 100
        bio_p = (len(bio_hits) / k) * 100
        status = "⚠️ COLLISION" if tech_p > 0 and bio_p > 0 else "✅ CLEAN"

        print(f"\n[QUERY] {q}")
        print(
            f"STATUS: {status} | AI Accuracy: {tech_p:.0f}% | Bio Noise: {bio_p:.0f}%"
        )
        print("=" * 100)

        for i, doc in enumerate(results):
            domain = doc.metadata.get("domain", "N/A")
            source = doc.metadata.get("source", "N/A")
            title = doc.metadata.get("title", "N/A")
            # Clean up content for display
            content = doc.page_content[:200].strip().replace("\n", " ") + "..."

            icon = "🤖" if domain == "technical" else "🧬"
            print(f"[{i+1}] {icon} [{domain.upper()}] {title}")
            print(f"    🔗 Source: {source}")
            print(f'    📜 Quote:  "{content}"')
            print("-" * 40)
        print()


collision_queries = [
    "How to control the agent's behavior and environment?",
    "The limitations of agent memory and reasoning.",
    "Mechanism of action for the primary agent.",
]

run_collision_diagnostic(collision_queries, vectorstore)


[QUERY] How to control the agent's behavior and environment?
STATUS: ⚠️ COLLISION | AI Accuracy: 60% | Bio Noise: 40%
[1] 🤖 [TECHNICAL] LLM Powered Autonomous Agents | Lil'Log
    🔗 Source: https://lilianweng.github.io/posts/2023-06-23-agent/
    📜 Quote:  "Prompt LM with 100 most recent observations and to generate 3 most salient high-level questions given a set of observations/statements. Then ask LM to answer those questions.   Planning & Reacting: tr..."
----------------------------------------
[2] 🧬 [BIOLOGICAL] Biological agent - Wikipedia
    🔗 Source: https://en.wikipedia.org/wiki/Biological_agent
    📜 Quote:  "Some biological agents have the ability to adversely affect human health in a variety of ways, ranging from relatively mild allergic reactions to serious medical conditions, including serious injury,..."
----------------------------------------
[3] 🤖 [TECHNICAL] LLM Powered Autonomous Agents | Lil'Log
    🔗 Source: https://lilianweng.github.io/posts/2023-06-23-agent/
 

## Phase 1: Knowledge Smelting (Background Mirror)

We will implement a `BackgroundSmelter` that runs in a separate thread to generate a **Global Intent Mirror (GIM)**.


In [ ]:
class GlobalIntentMirror(BaseModel):
    domain_name: str = Field(
        description="The primary professional domain of the knowledge base"
    )
    core_topics: list[str] = Field(
        description="Fixed list of 5-8 core technical or scientific topics"
    )
    key_terminology: dict[str, str] = Field(
        description="Specific definitions for ambiguous terms found in this index"
    )
    logical_bounds: str = Field(description="A summary of what is NOT in this database")


parser = JsonOutputParser(pydantic_object=GlobalIntentMirror)

In [ ]:
class BackgroundSmelter:
    def __init__(self, vectorstore, llm, cache_path="index_gim.json"):
        self.vectorstore = vectorstore
        self.llm = llm
        self.cache_path = cache_path
        self.mirror = None
        self._is_running = False

    def smelt(self):
        logger.info("[Smelter] Starting Knowledge Smelting process...")

        # Sample from both domains to understand the boundaries
        samples_a = self.vectorstore.similarity_search("LLM Agent", k=5)
        samples_b = self.vectorstore.similarity_search("Pathogen Virus", k=5)
        all_samples = samples_a + samples_b

        sample_text = "\n---\n".join(
            [
                f"Domain: {d.metadata.get('domain')}\nContent: {d.page_content[:300]}"
                for d in all_samples
            ]
        )

        template = """You are a Knowledge Engineer. Analyze the data fragments from a vector database 
and extract a logical mapping (Global Intent Mirror) of the knowledge base. 
Identify how terms like 'Agent', 'Memory', 'Environment' are used in THIS specific collection.

Data Fragments:
{samples}

{format_instructions}
"""
        prompt = ChatPromptTemplate.from_template(template)
        chain = prompt | self.llm | parser

        try:
            self.mirror = chain.invoke(
                {
                    "samples": sample_text,
                    "format_instructions": parser.get_format_instructions(),
                }
            )
            with open(self.cache_path, "w") as f:
                json.dump(self.mirror, f, indent=2)
            logger.success("[Smelter] Smelting complete. GIM cached.")
        except Exception as e:
            logger.error(f"[Smelter] Smelting failed: {str(e)}")

    def run_async(self):
        threading.Thread(target=self.smelt, daemon=True).start()


smelter_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
smelter = BackgroundSmelter(vectorstore, smelter_llm)
smelter.run_async()
logger.info("Background Smelter started.")

## Phase 2: Adaptive Domain Anchoring (Query Rewriting)

Using the cached GIM to resolve ambiguity before performing Multi-Query.


In [ ]:
class AdaptiveAnchoringChain:
    def __init__(self, llm, gim_path="index_gim.json"):
        self.llm = llm
        self.gim_path = gim_path

    def _get_gim(self):
        try:
            with open(self.gim_path, "r") as f:
                return json.load(f)
        except:
            return None

    def transform(self, question, k=3):
        gim = self._get_gim()
        if not gim:
            return [question]  # Fallback to original

        template = """You are an expert in {domain_name}. 
The current knowledge base covers: {core_topics}.
Note the following domain-specific terminology: {key_terminology}.

Your task is to generate {k} different technical variations of the user question to retrieve relevant documents.
Ensure the variations are strictly anchored to the {domain_name} context to avoid ambiguity.

Original question: {question}
Provide variations separated by newlines.
"""
        prompt = ChatPromptTemplate.from_template(template)
        chain = prompt | self.llm | StrOutputParser() | (lambda x: x.split("\n"))

        return chain.invoke(
            {
                "domain_name": gim["domain_name"],
                "core_topics": ", ".join(gim["core_topics"]),
                "key_terminology": json.dumps(gim["key_terminology"]),
                "k": k,
                "question": question,
            }
        )


anchor_llm = ChatOpenAI(model="gpt-4o", temperature=0.4)
anchoring_chain = AdaptiveAnchoringChain(anchor_llm)

## Phase 3: Evaluation (Head-to-Head Comparison)

We will test if the Adaptive Anchoring correctly filters out noisy results from the biological domain when asking about AI agents.


In [ ]:
test_queries = [
    "What are the limitations of an agent's memory?",
    "How does the environment affect the agent?",
    "Mechanism of action for the primary agent.",
]


def evaluate_recall(queries, retriever):
    all_docs = []
    for q in queries:
        all_docs.extend(retriever.invoke(q))

    # Calculate Domain Purity
    tech_count = len([d for d in all_docs if d.metadata.get("domain") == "technical"])
    bio_count = len([d for d in all_docs if d.metadata.get("domain") == "biological"])

    purity = (tech_count / len(all_docs)) * 100 if all_docs else 0
    return purity, tech_count, bio_count


retriever = vectorstore.as_retriever()

for query in test_queries:
    logger.info(f"--- Testing: {query} ---")

    # 1. Standard (Generic) Multi-Query Simulation
    generic_variations = [query, f"How to improve {query}", f"What is {query}"]
    g_purity, g_tech, g_bio = evaluate_recall(generic_variations, retriever)

    # 2. Adaptive Anchoring
    adaptive_variations = anchoring_chain.transform(query)
    a_purity, a_tech, a_bio = evaluate_recall(adaptive_variations, retriever)

    logger.info(f"[Generic]  Purity: {g_purity:.1f}% | Tech: {g_tech} | Bio: {g_bio}")
    logger.info(f"[Adaptive] Purity: {a_purity:.1f}% | Tech: {a_tech} | Bio: {a_bio}")
    logger.info(f"[Adaptive] Variations: {adaptive_variations}")